# Ablation Study — Hybrid Log Anomaly Detection
## Google Colab Runner

This notebook drives the full 6-stage ablation pipeline from `run_ablation.py`
on a Colab GPU/TPU instance.

**Workflow:**
1. Mount Google Drive and clone the repo (or copy data from Drive)
2. Install all dependencies
3. Verify GPU is available
4. Configure the ablation matrix (experiments to run)
5. Execute each experiment and collect metrics
6. Visualise and compare results across experiments

**Before running:**
- Upload `data/raw/BGL_full.log` (and/or `HDFS_full.log`) to your Google Drive
- Optionally upload pre-computed `data/processed/` artefacts to skip expensive stages
- Add your Azure OpenAI keys to Colab Secrets (key icon in the left sidebar) if LLM enrichment is enabled


## 1 — Environment Setup

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")


In [ ]:
# ── Launch settings and durable Drive artifact store ──────────────────────────
# This configuration trains and evaluates the base HDFS GAE from the prepared
# graph bundle. The reusable model and evaluation implementation live in
# run_ablation.py and src/modules/models/gae.py.
from pathlib import Path
import gzip
import json
import os
import shutil
import subprocess

REPOSITORY_URL = "https://github.com/michaail/hybrid-logs-analyzer.git"
GIT_REF = "main"
DATASET = "hdfs"
RUN_MODE = "train-only"
RUN_MATRIX = False               # Run only configs/ablation_base.yaml.
RUN_ID = "hdfs_base_clean_train"  # Change this to preserve results from another run.
INPUT_RUN_ID = None
GRAPH_DATASET_RELATIVE_PATH = (
    "data/processed/hdfs/"
    "20260818_0002_1_parser_3_graph_dataset.pt.gz"
)
REUSE_DRIVE_CACHE = True         # Pull completed cached stages before execution.

def find_local_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "modules" / "models" / "gae.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the web-app checkout (hybrid-logs-analyzer) "
        "after `git submodule update --init`."
    )

REPO_ROOT = Path("/content/magisterka-repo") if IN_COLAB else find_local_repository_root()
WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else REPO_ROOT
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    # github_token = userdata.get("GITHUB_TOKEN")
    clone_url = REPOSITORY_URL
    if REPOSITORY_URL.startswith("https://github.com/"):
        clone_url = REPOSITORY_URL

    if not REPO_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "--recurse-submodules", "--branch", GIT_REF, clone_url, str(REPO_ROOT)])
    else:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", GIT_REF])
    subprocess.check_call(["git", "-C", str(REPO_ROOT), "submodule", "update", "--init", "--depth", "1"])

    DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)

    def stage_from_drive(relative_path: str) -> Path:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        destination = WORKSPACE_ROOT / relative_path
        if not source.exists():
            raise FileNotFoundError(f"Missing Drive artifact: {source}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        return destination

    def stage_graph_dataset(relative_path: str) -> Path:
        staged_path = stage_from_drive(relative_path)
        if staged_path.suffix != ".gz":
            return staged_path

        graph_path = staged_path.with_suffix("")
        print(f"Decompressing {staged_path.name} → {graph_path.name}")
        with gzip.open(staged_path, "rb") as source, open(graph_path, "wb") as destination:
            shutil.copyfileobj(source, destination, length=16 * 1024 * 1024)
        staged_path.unlink()
        return graph_path

    def stage_tree_from_drive(relative_path: str) -> None:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        if source.exists():
            destination = WORKSPACE_ROOT / relative_path
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])

    if REUSE_DRIVE_CACHE:
        stage_tree_from_drive(f"artifacts/cache/{DATASET}")
        stage_tree_from_drive("artifacts/runs")

    if RUN_MODE == "full":
        stage_from_drive("data/raw/BGL_full.log" if DATASET == "bgl" else "data/raw/HDFS_full.log")
        if DATASET == "hdfs":
            stage_from_drive("data/raw/anomaly_label.csv")
    elif GRAPH_DATASET_RELATIVE_PATH:
        GRAPH_DATASET_PATH = stage_graph_dataset(GRAPH_DATASET_RELATIVE_PATH)
    elif INPUT_RUN_ID:
        run_manifest = stage_from_drive(f"artifacts/runs/{INPUT_RUN_ID}.json")
        graph_relative_path = json.loads(run_manifest.read_text())["artifacts"]["graph_dataset"]
        GRAPH_DATASET_PATH = stage_graph_dataset(graph_relative_path)
    else:
        raise ValueError("train-only mode needs INPUT_RUN_ID or GRAPH_DATASET_RELATIVE_PATH")

print(f"Code checkout : {REPO_ROOT}")
print(f"Workspace     : {WORKSPACE_ROOT}")

# Can be skipped

In [ ]:
# ── Install a PyG stack matching the runtime's preinstalled PyTorch ───────────
# requirements-colab.txt contains Python 3.13-compatible package versions.
# The local requirements.txt keeps the legacy Sentence Transformers stack
# required by the target device and is intentionally not used in Colab.
if IN_COLAB:
    subprocess.check_call([
        sys.executable, str(REPO_ROOT / "research" / "scripts" / "install_colab.py"),
        "--project-root", str(REPO_ROOT / "research"),
    ])
else:
    print("Local environment — install requirements.txt in your virtual environment.")

In [ ]:
# Dependency installation is consolidated in the preceding cell.


In [ ]:
# Dependency installation is consolidated in the preceding cell.


In [ ]:
# ── Activate the checked-out source and verify accelerator availability ─────────
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import torch
print(f"Working directory: {Path.cwd()}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Optional Azure credentials for LLM enrichment ─────────────────────────────
# Store these in Colab Secrets. They are loaded into this process only and are
# never copied to Drive or written to an experiment artifact.
AZURE_SECRET_KEYS = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT_MISTRAL_LARGE",
    "AZURE_OPENAI_DEPLOYMENT_MISTRAL_SMALL",
]
if IN_COLAB:
    loaded = []
    for secret_key in AZURE_SECRET_KEYS:
        secret_value = userdata.get(secret_key)
        if secret_value:
            os.environ[secret_key] = secret_value
            loaded.append(secret_key)
    print(f"Loaded {len(loaded)} Azure secrets from Colab Secrets.")
else:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
    print("Loaded local .env when present.")


---
## 2 — Versioned experiment configuration

The ablation matrix is committed in `configs/ablation_matrix.yaml`; edit and commit it with the code instead of changing notebook state. The launch cell selects the dataset and invokes the runner in a subprocess, which checkpoints every completed stage to Drive.


In [ ]:
# The matrix and defaults are versioned configuration, not notebook state.
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "ablation_base.yaml"
MATRIX_PATH = REPO_ROOT / "configs" / "ablation_matrix.yaml"
print(f"Base config: {BASE_CONFIG_PATH}")
print(f"Matrix     : {MATRIX_PATH if RUN_MATRIX else 'disabled'}")


---
## 3 — Run and checkpoint

The runner receives only explicit paths and configuration. After every completed stage it copies the durable cache manifest and payload to Drive. A later runtime stages that cache before execution, then reuses it when the configuration, inputs, and Git revision match.


In [ ]:
# ── Invoke the versioned runner ───────────────────────────────────────────────
runner_command = [
    sys.executable,
    str(REPO_ROOT / "run_ablation.py"),
    "--mode", RUN_MODE,
    "--config", str(BASE_CONFIG_PATH),
    "--workspace-root", str(WORKSPACE_ROOT),
    "--code-root", str(REPO_ROOT),
    "--set", f"experiment.dataset={DATASET}",
]
if RUN_MATRIX:
    runner_command.extend(["--matrix", str(MATRIX_PATH)])
if RUN_ID:
    runner_command.extend(["--run-id", RUN_ID])
if RUN_MODE == "train-only":
    runner_command.extend(["--graph-dataset", str(GRAPH_DATASET_PATH)])
if IN_COLAB:
    runner_command.extend(["--checkpoint-root", str(DRIVE_ARTIFACT_ROOT)])

def final_drive_sync() -> None:
    if not IN_COLAB:
        return
    for directory_name in ("artifacts", "models", "outputs", "runs"):
        source = WORKSPACE_ROOT / directory_name
        if source.exists():
            destination = DRIVE_ARTIFACT_ROOT / directory_name
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"], check=True)

try:
    completed = subprocess.run(runner_command, check=False)
finally:
    # Covers final reports and any completed stage when the runner exits early.
    final_drive_sync()

if completed.returncode:
    raise RuntimeError(f"Experiment runner failed with exit code {completed.returncode}")

if RUN_MATRIX:
    result_files = sorted(
        (WORKSPACE_ROOT / "outputs" / DATASET).glob("*_ablation_matrix/ablation_results.json"),
        key=lambda path: path.stat().st_mtime,
    )
    RESULTS_JSON = result_files[-1]
    all_results = json.loads(RESULTS_JSON.read_text())
else:
    run_metrics = sorted(
        (WORKSPACE_ROOT / "outputs" / DATASET).glob("*/metrics.json"),
        key=lambda path: path.stat().st_mtime,
    )[-1]
    all_results = [{"name": RUN_ID or run_metrics.parent.name, "status": "OK", **json.loads(run_metrics.read_text())}]

print(f"Results: {RESULTS_JSON if RUN_MATRIX else run_metrics}")


---
## 4 — Results Table

In [ ]:
import pandas as pd
import json
from pathlib import Path

# Build summary DataFrame
metric_cols = ["test_f1", "test_pr_auc", "test_roc_auc", "val_pr_auc", "val_roc_auc"]
display_cols = ["name", "dataset", "elapsed_min", "status"] + metric_cols

df_results = pd.DataFrame(all_results)

# Round metric columns
for col in metric_cols:
    if col in df_results.columns:
        df_results[col] = df_results[col].round(4)

# Display — highlight best metric per column
ok_mask = df_results["status"] == "OK"

def _highlight_best(col):
    if col.name not in metric_cols or not ok_mask.any():
        return [""] * len(col)
    best_val = col[ok_mask].max()
    return ["font-weight: bold; color: #2a7ae2" if (ok_mask.iloc[i] and v == best_val) else ""
            for i, v in enumerate(col)]

display_df = df_results[[c for c in display_cols if c in df_results.columns]]
display_df.style.apply(_highlight_best)


In [ ]:
# The runner already checkpoints raw metrics. These notebook-only exports (table
# and plots) are written directly to the durable Drive store.
from datetime import datetime
report_root = DRIVE_ARTIFACT_ROOT if IN_COLAB else WORKSPACE_ROOT
outputs_dir = report_root / "outputs" / DATASET / "notebook_reports"
outputs_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_csv = outputs_dir / f"{timestamp}_ablation_results.csv"
results_json = outputs_dir / f"{timestamp}_ablation_results.json"

df_results.to_csv(results_csv, index=False)
results_json.write_text(json.dumps(all_results, indent=2, default=str))
print(f"Notebook report saved: {results_json}")


---
## 5 — Visualisation

Compare experiments across the three key metrics.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

ok_df = df_results[df_results["status"] == "OK"].copy()

if ok_df.empty:
    print("No successful experiments to plot.")
else:
    metrics_to_plot = [
        ("test_f1",       "Test F1"),
        ("test_pr_auc",   "Test PR-AUC"),
        ("test_roc_auc",  "Test ROC-AUC"),
    ]
    metrics_to_plot = [(m, lbl) for m, lbl in metrics_to_plot if m in ok_df.columns]

    n_metrics = len(metrics_to_plot)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5), sharey=False)
    if n_metrics == 1:
        axes = [axes]

    fig.suptitle(
        f"Ablation Study Results — {DATASET.upper()}",
        fontsize=14, fontweight="bold", y=1.02,
    )

    COLORS = plt.cm.tab10.colors
    names = ok_df["name"].tolist()
    x = np.arange(len(names))

    for ax, (metric, label) in zip(axes, metrics_to_plot):
        vals = ok_df[metric].tolist()
        bars = ax.barh(x, vals, color=COLORS[:len(names)], edgecolor="white", height=0.6)

        # Bold bar for the best
        best_idx = int(np.argmax(vals))
        bars[best_idx].set_edgecolor("#111")
        bars[best_idx].set_linewidth(2)

        # Value labels
        for bar, v in zip(bars, vals):
            ax.text(
                v + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{v:.4f}", va="center", fontsize=8.5,
            )

        ax.set_yticks(x)
        ax.set_yticklabels(names, fontsize=9)
        ax.set_xlabel(label, fontsize=10)
        ax.set_xlim(0, max(vals) * 1.15 if vals else 1)
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
        ax.grid(axis="x", alpha=0.3)
        ax.invert_yaxis()   # best (top of table) at top of chart

    plt.tight_layout()
    fig_path = outputs_dir / f"{timestamp}_ablation_comparison.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure saved → {fig_path}")


In [ ]:
# ── Training-loss curves per experiment ──────────────────────────────────────
# Pulls history from all_results (populated in the run cell above).

histories = {
    r["name"]: r["history"]
    for r in all_results
    if r.get("status") == "OK" and "history" in r
}

if not histories:
    print("No training histories available.")
else:
    n_exp = len(histories)
    fig, axes = plt.subplots(1, n_exp, figsize=(5 * n_exp, 4), sharey=False)
    if n_exp == 1:
        axes = [axes]

    fig.suptitle("Training Loss Curves", fontsize=13, fontweight="bold", y=1.02)

    for ax, (exp_name, hist) in zip(axes, histories.items()):
        epochs = range(1, len(hist["total"]) + 1)
        ax.plot(epochs, hist["total"],     "k-",  lw=2, label="Total")
        ax.plot(epochs, hist["structure"], "b--", lw=1.4, label="Structure")
        ax.plot(epochs, hist["node"],      "g-.", lw=1.4, label="Node")
        ax.plot(epochs, hist["edge"],      "r:",  lw=1.4, label="Edge")
        ax.set_title(exp_name, fontsize=9)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    loss_fig_path = outputs_dir / f"{timestamp}_loss_curves.png"
    plt.savefig(loss_fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure saved → {loss_fig_path}")


---
## 6 — Load & Replay Previous Results (Optional)

Use this section to reload results from a previous Colab session without re-running experiments.


In [ ]:
# ── Reload from a saved JSON ──────────────────────────────────────────────────
# Uncomment and set the correct filename to replay results without re-running.
#
# import json, pandas as pd
# from pathlib import Path
#
# RESULTS_FILE = Path(REPO_ROOT) / "outputs" / "20260708_1030_ablation_results.json"
# with open(RESULTS_FILE) as fh:
#     all_results = json.load(fh)
#
# df_results = pd.DataFrame(all_results)
# print(f"Loaded {len(all_results)} results from {RESULTS_FILE}")
# display(df_results[["name", "dataset", "test_f1", "test_pr_auc", "test_roc_auc", "status"]])
